# 02 Preprocessing

Objective: Handle missing values, scaling, and initial data cleaning.

In [18]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score, StratifiedKFold

In [5]:
df = pd.read_csv(r'C:\Users\David\BTC-trend-prediction\data\raw\coin_Bitcoin.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

df.drop(columns=['SNo','Name','Symbol'], inplace=True)
df['Daily_Returns'] = df['Close'].pct_change()
df = df.dropna()

In [6]:
df['H+1_Return'] = df['Daily_Returns'].shift(-1)
#Shift the returns up by 1 row to align today's features with tomorrow's outcome.

In [8]:
# Apply your 0.5% threshold logic. 
# If next day > 0.005, make it a 1. Otherwise, make it a 0.
df['Target'] = (df['H+1_Return'] > 0.005).astype(int)

In [9]:
df = df.dropna()

In [10]:
X = df[['Open','High','Low','Close','Volume', 'Daily_Returns']]
y = df['Target']

In [20]:
#Data Splitting via timeseries
tscv = TimeSeriesSplit(n_splits=5)

In [21]:
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

In [23]:
train_end_date = df.loc[X_train.index[-1], 'Date']
test_start_date = df.loc[X_test.index[0], 'Date']

print(f"\nTraining data ends on: {train_end_date}")
print(f"Testing data starts on: {test_start_date}")

#just a simple verification


Training data ends on: 2020-02-23 23:59:59
Testing data starts on: 2020-02-24 23:59:59


In [24]:
scaler = StandardScaler()

# Fit on Train ONLY. Transform both.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)